# Solutions · 05 · Nonlinear diffusion

Worked solutions to the exercises of [notebook 05](../notebooks/05_nonlinear_diffusion.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmolines is missing): install it from GitHub
    import engmolines
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmolines"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engmolines import problems as pr, integrators as ti, grids, boundary as bc, operators as op
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Run the porous-medium problem with m = 3 and m = 4 (the Barenblatt exponents change: alpha = beta = 1/(m+1), kk = (m-1)/(2m(m+1))) and confirm the front speed law t^(1/(m+1)).

In [2]:
for m in (3, 4):
    p = pr.porous_medium(n=400, m=m)
    s = p.solve("bdf", 2e-3, order=2, n_out=5)
    edges = [p.x[y > 1e-6][-1] for y in s.y]
    t0 = 0.1
    fit = np.polyfit(np.log(s.t + t0), np.log(edges), 1)[0]
    print(f"m = {m}: error vs Barenblatt {p.error(s):.1e}; front position ~ t^{fit:.3f} (theory 1/(m+1) = {1 / (m + 1):.3f})")

m = 3: error vs Barenblatt 8.4e-03; front position ~ t^0.253 (theory 1/(m+1) = 0.250)


m = 4: error vs Barenblatt 3.0e-02; front position ~ t^0.204 (theory 1/(m+1) = 0.200)


The Barenblatt profile's exponent and the front law t^(1/(m+1)) change with m, and the discretisation
follows them: the measured exponent of the front position matches the theory to the second decimal for both.

## Exercise 2

Perturb the Allen-Cahn kink by adding 0.01 sin(pi x) and integrate to t = 50. Does the kink return to its shape, and does it drift? Measure the drift against eps.

In [3]:
eps = 0.01
p = pr.allen_cahn(eps=eps)
def zero_crossing(y):
    i = int(np.argmax(y > 0))
    return p.x[i - 1] - y[i - 1] * (p.x[i] - p.x[i - 1]) / (y[i] - y[i - 1])
for name, pert in (("odd perturbation 0.01 sin(pi x)", 0.01 * np.sin(np.pi * p.x)), ("even perturbation 0.01 cos(pi x / 2)", 0.01 * np.cos(np.pi * p.x / 2))):
    s = ti.bdf(p.rhs, p.y0 + pert, (0, 50.0), 0.05, 2, band=(1, 1), n_out=10)
    pos = np.array([zero_crossing(y) for y in s.y])
    shape_err = [np.max(np.abs(y - np.tanh((p.x - x0) / np.sqrt(2 * eps)))) for y, x0 in zip(s.y, pos)]
    print(f"{name}: kink position {['%.1e' % v for v in np.round(pos[::2], 12)]}; deviation from a shifted kink {['%.1e' % e for e in shape_err[::2]]}")

odd perturbation 0.01 sin(pi x): kink position ['0.0e+00', '0.0e+00', '0.0e+00', '-0.0e+00', '-0.0e+00', '-0.0e+00']; deviation from a shifted kink ['1.0e-02', '2.8e-04', '2.8e-04', '2.8e-04', '2.8e-04', '2.8e-04']


even perturbation 0.01 cos(pi x / 2): kink position ['-1.4e-03', '-2.1e-03', '-2.1e-03', '-2.1e-03', '-2.1e-03', '-2.1e-03']; deviation from a shifted kink ['8.4e-03', '2.9e-04', '2.9e-04', '2.9e-04', '2.9e-04', '2.9e-04']


Both perturbations decay in a time of order one (the stable eigenvalues of the kink) and the profile returns
to the tanh shape to the grid's accuracy. The kink's position is a neutral mode: it shifts by the
perturbation's projection on the kink's derivative and then stays. The odd perturbation sin πx is orthogonal
to that even mode and moves the kink by nothing (10⁻¹⁷); the even perturbation shifts it, and the shift is
never undone, because nothing restores a neutral mode. The settled position's constancy over 50 time units
is the test that the scheme adds no spurious drift of its own.

## Exercise 3

**Implement it yourself:** write the Cahn-Hilliard right-hand side as two applications of the 3-point Laplacian with no-flux ghosts, and verify mass conservation and the band (2, 2) of its Jacobian with `linalg.banded_jacobian`.

In [4]:
p = pr.cahn_hilliard()
g, eps = p.grid, 0.02
lap = lambda v: op.diffusion(v, g, 1.0, bc.Neumann(0.0), bc.Neumann(0.0))   # noqa: E731
rhs_hand = lambda t, u: lap(u**3 - u - eps**2 * lap(u))                      # noqa: E731
u = p.y0
print("hand-written vs library right-hand side:", f"{np.max(np.abs(rhs_hand(0, u) - p.rhs(0, u))):.1e}")
from engmolines import linalg as la
Jb = la.from_banded(la.banded_jacobian(lambda v: rhs_hand(0, v), u, 2, 2), 2, 2)
Jd = la.dense_jacobian(lambda v: rhs_hand(0, v), u)
print("band (2, 2) captures the Jacobian:", f"{np.max(np.abs(Jb - Jd)):.1e}", "; rates sum to", f"{np.sum(rhs_hand(0, u) * g.widths):.1e}", "(mass conserved)")

hand-written vs library right-hand side: 0.0e+00
band (2, 2) captures the Jacobian: 0.0e+00 ; rates sum to -5.7e-14 (mass conserved)


Two applications of the three-point Laplacian give a five-point operator, hence the pentadiagonal Jacobian;
the no-flux ghosts make each application conservative, so the composite is too. The library's right-hand
side is this composition.